
**Embedding Input Size Analysis**<br>
Counts and plots the size of each input that would be sent to Cohere embed v4

In [ ]:
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

from utils.image_handling import get_image_paths
from utils.xml_parser import PaperParser, Paper, Chunk, MEDIA_MARKER
from config import DATASET_DIR, COHERE_TRANSFORMABLE_FORMATS, BATCH_MAX_TOKENS

DATASET = DATASET_DIR / "papers"

# %%
def measure_input(chunk: Chunk, paper_dir: Path) -> dict:
    """
    mirrors chunk_to_input() but measures sizes instead of building the payload.
    returns a dict with text_chars, n_images, approx_bytes, and input_type.
    """
    text = f"Section: {chunk.section} Content: {chunk.text}"

    if not MEDIA_MARKER.search(text):
        return {"text_chars": len(text), "n_images": 0, "approx_bytes": len(text.encode()), "input_type": "text"}

    text_chars = 0
    n_images = 0
    image_bytes = 0

    last = 0
    for m in MEDIA_MARKER.finditer(text):
        before = text[last:m.start()].strip()
        text_chars += len(before)

        media_path = paper_dir / m.group(1)
        paths = get_image_paths(media_path)
    
        for p in paths:
            try:
                image_bytes += p.stat().st_size
                n_images += 1
            except FileNotFoundError:
                pass

        last = m.end()

    text_chars += len(text[last:].strip())

    return {
        "text_chars": text_chars,
        "n_images": n_images,
        "approx_bytes": len(text.encode()) + image_bytes,
        "input_type": "interleaved" if n_images > 0 else "text",
    }


def make_batches(paper: Paper, max_tokens: int = BATCH_MAX_TOKENS) -> list[list[Chunk]]:
    batches, current, current_tokens = [], [], 0
    for chunk in (paper.abstract + paper.body):
        if current_tokens + chunk.n_tokens > max_tokens:
            if current:
                batches.append(current)
            current, current_tokens = [chunk], chunk.n_tokens
        else:
            current.append(chunk)
            current_tokens += chunk.n_tokens
    if current:
        batches.append(current)
    return batches


# %%
records = []

for paper_dir in sorted(DATASET.iterdir()):
    if paper_dir.name == ".DS_Store":
        continue
    xml_files = list(paper_dir.glob("*.xml"))
    if not xml_files:
        print(f"No XML in {paper_dir.name}, skipping")
        continue
    try:
        parser = PaperParser(xml=xml_files[0])
        paper = parser.parse_paper()
        if paper.title == "An Open-Source Reproducible Workflow for Pocket-Oriented Virtual Screening and ADME-Integrated Chemoinformatics: A Multi-Target Flavivirus Case Study":
            print(f"Found: {paper_dir}")
    except Exception as e:
        print(f"Parse error {paper_dir.name}: {e}")
        continue

    batches = make_batches(paper)
    for batch_idx, batch in enumerate(batches):
        for chunk in batch:
            m = measure_input(chunk, paper_dir)
            records.append({
                "paper": paper_dir.name,
                "batch": batch_idx,
                "section": chunk.section,
                "n_tokens": chunk.n_tokens,
                "text": chunk.text,
                **m,
            })

print(f"Total inputs measured: {len(records)}")

# %%
approx_bytes  = [r["approx_bytes"]  for r in records]
text_chars    = [r["text_chars"]    for r in records]
n_images      = [r["n_images"]      for r in records]
input_types   = [r["input_type"]    for r in records]
n_tokens      = [r["n_tokens"]      for r in records]

type_counts = defaultdict(int)
for t in input_types:
    type_counts[t] += 1

print(f"text-only:   {type_counts['text']:>6}")
print(f"interleaved: {type_counts['interleaved']:>6}")
print(f"\napprox_bytes  — mean: {np.mean(approx_bytes):>10,.0f}  max: {np.max(approx_bytes):>12,.0f}")
print(f"text_chars    — mean: {np.mean(text_chars):>10,.0f}  max: {np.max(text_chars):>12,.0f}")
print(f"n_images      — mean: {np.mean(n_images):>10.2f}  max: {np.max(n_images):>12}")
print(f"n_tokens      — mean: {np.mean(n_tokens):>10.2f}  max: {np.max(n_tokens):>12}")

# %%
max_token_record = max(records, key=lambda r: r["n_tokens"])
max_image_record = max(records, key=lambda r: r["n_images"])

print("=== Max tokens ===")
print(f"paper:    {max_token_record['paper']}")
print(f"section:  {max_token_record['section']}")
print(f"n_tokens: {max_token_record['n_tokens']}")
print(f"raw chunk: {max_token_record['text']}")

print("\n=== Max images ===")
print(f"paper:   {max_image_record['paper']}")
print(f"section: {max_image_record['section']}")
print(f"n_images: {max_image_record['n_images']}")
print(f"raw chunk: {max_image_record['text']}")

# %%
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Embedding Input Size Distribution", fontsize=15, fontweight="bold", y=1.01)

colors = {"text": "#4C9BE8", "interleaved": "#E8844C"}

# 1. approx payload bytes by type
ax = axes[0, 0]
for t, c in colors.items():
    vals = [r["approx_bytes"] for r in records if r["input_type"] == t]
    if vals:
        ax.hist(vals, bins=50, alpha=0.7, color=c, label=t)
ax.set_title("Approx Payload Bytes")
ax.set_xlabel("bytes")
ax.set_ylabel("count")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}MB" if x >= 1e6 else f"{x/1e3:.0f}KB"))
ax.legend()

# 2. token count distribution
ax = axes[0, 1]
ax.hist(n_tokens, bins=50, color="#6BCB77", edgecolor="white", linewidth=0.3)
ax.axvline(BATCH_MAX_TOKENS, color="red", linestyle="--", linewidth=1.2, label=f"batch limit ({BATCH_MAX_TOKENS})")
ax.set_title("Token Count per Chunk")
ax.set_xlabel("tokens")
ax.set_ylabel("count")
ax.legend()

# 3. images per input (interleaved only)
ax = axes[1, 0]
img_counts = [r["n_images"] for r in records if r["input_type"] == "interleaved"]
if img_counts:
    max_imgs = max(img_counts)
    bins = range(0, max_imgs + 2)
    ax.hist(img_counts, bins=bins, color="#E8844C", edgecolor="white", linewidth=0.3, align="left")
    ax.set_xticks(range(0, max_imgs + 1))
ax.set_title("Images per Interleaved Input")
ax.set_xlabel("n images")
ax.set_ylabel("count")

# 4. text chars distribution
ax = axes[1, 1]
for t, c in colors.items():
    vals = [r["text_chars"] for r in records if r["input_type"] == t]
    if vals:
        ax.hist(vals, bins=50, alpha=0.7, color=c, label=t)
ax.set_title("Text Characters per Input")
ax.set_xlabel("chars")
ax.set_ylabel("count")
ax.legend()

plt.tight_layout()
plt.savefig("embed_input_sizes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved embed_input_sizes.png")